In [ ]:
!pip install -q google-genai pydantic
import json, time
import pandas as pd
from google import genai
from google.genai import types

In [ ]:
from getpass import getpass
MODEL_NAME = "gemini-4-hydrogen"
API_KEY = getpass("AQ.Ab8RN6LI5KWY8PFl80l0LDH4qE6X4MTobFGmSSGIPMR2M_S5fA ")
client = genai.Client(api_key=API_KEY)
print("client ready")

AQ.Ab8RN6LI5KWY8PFl80l0LDH4qE6X4MTobFGmSSGIPMR2M_S5fA ··········
client ready


In [ ]:
r = client.models.generate_content(model=MODEL_NAME, contents="Say hello in one word.")
print(r.text)

Hello


In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = "You are a careful analyst who classifies call-centre notes for an outpatient clinic group."

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

def build_prompt(note_text):
    return f"""Context: CareLoop Clinics phones patients who missed an appointment and writes a short note.
Role: You are an operations analyst reading one call-centre note.
Action: Identify the single main reason the patient missed the visit. root_cause must be exactly one of: {', '.join(ALLOWED)}. If the note is unclear or fits none of them, answer "other" rather than guessing. Give a confidence between 0 and 1, a summary of at most 25 words, and one recommended_action.
Tone: Neutral and factual.
Format: Return only JSON with the keys root_cause, confidence, summary, recommended_action.

Note: {note_text}"""

def analyse_note(note_text):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="low")))
    return json.loads(response.text)

In [ ]:
#task 4.2
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(root_cause):
    if root_cause not in ALLOWED:
        raise ValueError(f"root_cause '{root_cause}' not in allowed list")

In [ ]:
import pandas as pd

notes = pd.read_csv("followup_notes.csv")

print("CSV loaded successfully!")
print(notes.head())

CSV loaded successfully!
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [ ]:
def call_raw(note_text):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="low")))
    return response.text

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
rows, api_calls = [], 0

for _, r in notes.head(10).iterrows():
    note_id, appt_id, text = r["note_id"], r["appointment_id"], r["note_text"]
    rec = {"note_id": note_id, "appointment_id": appt_id, "root_cause": "", "confidence": "",
           "summary": "", "recommended_action": "", "validation_status": "", "raw_response": ""}

    if pd.isna(text) or str(text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank")
        rows.append(rec)
        continue

    try:
        raw = call_raw(text)
        api_calls += 1
        rec["raw_response"] = raw
        data = json.loads(raw)
        for f in FIELDS:                       # copy whatever fields exist, repair nothing
            if f in data:
                rec[f] = data[f]
        FollowupInsight(**data)
        check_allowed(data["root_cause"])
        rec["validation_status"] = "valid"
    except Exception as e:
        rec["validation_status"] = "invalid"
        print(note_id, "invalid:", e)
    rows.append(rec)
    time.sleep(13)

out = pd.DataFrame(rows)
out.to_csv("notes_analysed.csv", index=False)
print("API calls in 4.2:", api_calls)
print(out[["note_id", "root_cause", "confidence", "validation_status"]])

N001 invalid: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-4-hydrogen, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 5m34.338150659s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.go

In [ ]:
todo = out.index[(out["raw_response"] == "") & (out["validation_status"] != "skipped_blank")]
print("retrying:", list(out.loc[todo, "note_id"]))

new_calls = 0
for i in todo:
    text = notes.loc[notes["note_id"] == out.loc[i, "note_id"], "note_text"].iloc[0]
    try:
        raw = call_raw(text)
    except Exception as e:
        print(out.loc[i, "note_id"], "API error, stopping:", str(e)[:150])
        break
    new_calls += 1
    out.loc[i, "raw_response"] = raw
    try:
        data = json.loads(raw)
        for f in FIELDS:
            if f in data:
                out.loc[i, f] = data[f]
        FollowupInsight(**data)
        check_allowed(data["root_cause"])
        out.loc[i, "validation_status"] = "valid"
    except Exception as e:
        out.loc[i, "validation_status"] = "invalid"
        print(out.loc[i, "note_id"], "invalid:", e)
    time.sleep(13)

print("successful calls in this retry:", new_calls)
print(out[["note_id", "root_cause", "confidence", "validation_status"]])

retrying: ['N001', 'N002', 'N003', 'N005', 'N006', 'N007', 'N008']
N001 API error, stopping: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 
successful calls in this retry: 0
  note_id        root_cause confidence validation_status
0    N001                                        invalid
1    N002                                        invalid
2    N003                                        invalid
3    N004             other        0.0     skipped_blank
4    N005                                        invalid
5    N006                                        invalid
6    N007                                        invalid
7    N008                                        invalid
8    N009             other       0.95             valid
9    N010  work_or_schedule       0.95             valid
